In [27]:
import pandas as pd
import numpy as np


df.columns.tolist()


['application_id',
 'submission_date',
 'application_status',
 'district_id',
 'proposed_use',
 'max_storeys',
 'residential_units',
 'non_residential_gfa_m2',
 'inside_mtsa',
 'mtsa_corridor',
 'distance_to_transit_m',
 'district_name',
 'approval_score',
 'synthetic_approval',
 'property_value_change_pct',
 'default_risk_score',
 'capital_concentration_risk',
 'construction_loan_feasibility',
 'insurance_premium_shift_index',
 'rent_increase_forecast_pct',
 'rent_burden_score',
 'displacement_risk_score',
 'transit_speed_impact_score']

In [28]:
df.columns = (
    df.columns
    .str.strip()              # remove leading/trailing spaces
    .str.lower()              # lowercase
    .str.replace(" ", "_")    # replace spaces with _
)

In [29]:
df.columns

Index(['application_id', 'submission_date', 'application_status',
       'district_id', 'proposed_use', 'max_storeys', 'residential_units',
       'non_residential_gfa_m2', 'inside_mtsa', 'mtsa_corridor',
       'distance_to_transit_m', 'district_name', 'approval_score',
       'synthetic_approval', 'property_value_change_pct', 'default_risk_score',
       'capital_concentration_risk', 'construction_loan_feasibility',
       'insurance_premium_shift_index', 'rent_increase_forecast_pct',
       'rent_burden_score', 'displacement_risk_score',
       'transit_speed_impact_score'],
      dtype='object')

In [30]:
import pandas as pd
import numpy as np

# Reload data
df = pd.read_csv("mississauga_planning_clean.csv")

# Clean column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

df.head()

,application_id,submission_date,application_status,district_id,proposed_use,max_storeys,residential_units,non_residential_gfa_m2,inside_mtsa,mtsa_corridor,distance_to_transit_m,district_name
0,SPM 22 107,2022-07-29,Withheld,1.0,General Retail Commercial,2.0,0,158.38,True,Dundas BRT,72,Lakeview / Port Credit
1,SP 22 10,2022-01-19,Withheld,5.0,General Retail Commercial,6.0,0,10175.00,False,Not within a Major Transit Station Area.,176,Malton / Northeast
2,SP 22 103,2022-07-20,Withheld,1.0,Residential Townhouses,3.0,70,0.00,False,Not within a Major Transit Station Area.,246,Lakeview / Port Credit
3,SP 23 2,2023-01-17,Withheld,9.0,Residential Townhouses; Residential Apartments,12.0,212,0.00,False,Not within a Major Transit Station Area.,70,Meadowvale Village
4,OZ/OPA 22 5,2022-01-31,Appealed,1.0,Residential Apartments; General Retail Commercial,NaN,42,150.00,True,Hurontario LRT,60,Lakeview / Port Credit


### CREATION OF THE SYNTHETIC DATA

In [31]:
df["inside_mtsa"] = df["inside_mtsa"].astype(str).str.upper().map({"TRUE":1, "FALSE":0})

df["max_storeys"] = pd.to_numeric(df["max_storeys"], errors="coerce")

df["residential_units"] = pd.to_numeric(df["residential_units"], errors="coerce").fillna(0)

df["non_residential_gfa_m2"] = pd.to_numeric(
    df["non_residential_gfa_m2"], errors="coerce"
).fillna(0)

df["distance_to_transit_m"] = pd.to_numeric(
    df["distance_to_transit_m"], errors="coerce"
)

In [6]:
score = np.zeros(len(df))

# MTSA boost
score += 25 * df["inside_mtsa"]

# Transit distance logic
score += np.where(df["distance_to_transit_m"] <= 500, 15, 0)
score += np.where((df["distance_to_transit_m"] > 500) & 
                  (df["distance_to_transit_m"] <= 1000), 5, 0)
score -= np.where(df["distance_to_transit_m"] > 1500, 10, 0)

# Height logic
score -= np.where((df["max_storeys"] > 25) & (df["inside_mtsa"] == 0), 15, 0)
score += np.where((df["max_storeys"] >= 6) & 
                  (df["max_storeys"] <= 20), 5, 0)

# Residential scale
score += np.where(df["residential_units"] > 200, 5, 0)

# Large GFA outside MTSA penalty
score -= np.where(
    (df["non_residential_gfa_m2"] > 10000) &
    (df["inside_mtsa"] == 0),
    10,
    0
)

# Mixed use boost
score += np.where(
    df["proposed_use"].str.contains("Residential", na=False) &
    df["proposed_use"].str.contains("Commercial", na=False),
    8,
    0
)

In [7]:
df["approval_score"] = np.clip(50 + score, 0, 100)

prob = np.clip(df["approval_score"] / 100, 0.05, 0.95)

df["synthetic_approval"] = np.random.binomial(1, prob)

df[["approval_score", "synthetic_approval"]].head()

,approval_score,synthetic_approval
0,90.0,1
1,60.0,0
2,65.0,1
3,75.0,1
4,98.0,1


In [8]:
df.to_csv("mississauga_planning_mockdata_v1.csv", index=False)

In [9]:
df_mock = pd.read_csv("mississauga_planning_mockdata_v1.csv")

df_mock.head()

,application_id,submission_date,application_status,district_id,proposed_use,max_storeys,residential_units,non_residential_gfa_m2,inside_mtsa,mtsa_corridor,distance_to_transit_m,district_name,approval_score,synthetic_approval
0,SPM 22 107,2022-07-29,Withheld,1.0,General Retail Commercial,2.0,0,158.38,1,Dundas BRT,72,Lakeview / Port Credit,90.0,1
1,SP 22 10,2022-01-19,Withheld,5.0,General Retail Commercial,6.0,0,10175.00,0,Not within a Major Transit Station Area.,176,Malton / Northeast,60.0,0
2,SP 22 103,2022-07-20,Withheld,1.0,Residential Townhouses,3.0,70,0.00,0,Not within a Major Transit Station Area.,246,Lakeview / Port Credit,65.0,1
3,SP 23 2,2023-01-17,Withheld,9.0,Residential Townhouses; Residential Apartments,12.0,212,0.00,0,Not within a Major Transit Station Area.,70,Meadowvale Village,75.0,1
4,OZ/OPA 22 5,2022-01-31,Appealed,1.0,Residential Apartments; General Retail Commercial,NaN,42,150.00,1,Hurontario LRT,60,Lakeview / Port Credit,98.0,1


From the data obtained from the city of Mississauga above, we did not have enough data to build our demo. Our only logical route was to create Approval scores with the Assistant of Ai below.

Although the data is Synthetic, it is structurally modeled on real infrasctructure thresholds and polixy frameworks obtained in the above data.

In [10]:
import matplotlib.pyplot as plt
import numpy as np

def plot_gauge(score, threshold=60):

    fig, ax = plt.subplots(figsize=(6,4), subplot_kw={'projection': 'polar'})

    # Hide full circle
    ax.set_theta_offset(np.pi)
    ax.set_theta_direction(-1)
    ax.set_ylim(0, 1)

    # Remove grid and labels
    ax.set_xticks([])
    ax.set_yticks([])
    ax.spines['polar'].set_visible(False)

    # Create zones
    theta = np.linspace(0, np.pi, 100)

    # Red zone (decline)
    red_zone = np.linspace(0, np.pi*(threshold/100), 100)
    ax.fill_between(red_zone, 0, 1, alpha=0.4)

    # Green zone (approve)
    green_zone = np.linspace(np.pi*(threshold/100), np.pi, 100)
    ax.fill_between(green_zone, 0, 1, alpha=0.2)

    # Needle
    needle_angle = np.pi * (score/100)
    ax.plot([needle_angle, needle_angle], [0, 1])

    status = "APPROVED" if score >= threshold else "DECLINED"

    plt.title(f"{status} — Score: {score}/100")
    plt.show()

In [11]:
df_mock = pd.read_csv("mississauga_planning_mockdata_v1.csv")

df_mock.head()

,application_id,submission_date,application_status,district_id,proposed_use,max_storeys,residential_units,non_residential_gfa_m2,inside_mtsa,mtsa_corridor,distance_to_transit_m,district_name,approval_score,synthetic_approval
0,SPM 22 107,2022-07-29,Withheld,1.0,General Retail Commercial,2.0,0,158.38,1,Dundas BRT,72,Lakeview / Port Credit,90.0,1
1,SP 22 10,2022-01-19,Withheld,5.0,General Retail Commercial,6.0,0,10175.00,0,Not within a Major Transit Station Area.,176,Malton / Northeast,60.0,0
2,SP 22 103,2022-07-20,Withheld,1.0,Residential Townhouses,3.0,70,0.00,0,Not within a Major Transit Station Area.,246,Lakeview / Port Credit,65.0,1
3,SP 23 2,2023-01-17,Withheld,9.0,Residential Townhouses; Residential Apartments,12.0,212,0.00,0,Not within a Major Transit Station Area.,70,Meadowvale Village,75.0,1
4,OZ/OPA 22 5,2022-01-31,Appealed,1.0,Residential Apartments; General Retail Commercial,NaN,42,150.00,1,Hurontario LRT,60,Lakeview / Port Credit,98.0,1


In [12]:
import numpy as np
import pandas as pd

np.random.seed(42)

df = df_mock.copy()

# --- Safety: ensure required columns exist ---
required = ["residential_units","non_residential_gfa_m2","max_storeys","distance_to_transit_m","inside_mtsa"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# --- Helpers ---
def clip01(x): 
    return np.clip(x, 0, 1)

def minmax(series):
    s = pd.to_numeric(series, errors="coerce")
    mn, mx = np.nanmin(s), np.nanmax(s)
    if mn == mx:
        return pd.Series(np.zeros(len(s)), index=series.index)
    return (s - mn) / (mx - mn)

# =========================
# 1) Projected population increase (people)
# =========================
# Planning-style assumption: persons per unit.
# Use a small realistic range and draw per-row to avoid a single constant.
ppu = np.random.uniform(2.2, 2.8, len(df))  # assumption band (document it)
df["projected_population_increase"] = (df["residential_units"].fillna(0) * ppu).round(0).astype(int)

# =========================
# 2) Transit rider impact (proxy)
# =========================
# More units and closer transit => higher rider impact.
# MTSA boosts propensity.
dist = pd.to_numeric(df["distance_to_transit_m"], errors="coerce").fillna(df["distance_to_transit_m"].median())
units = pd.to_numeric(df["residential_units"], errors="coerce").fillna(0)

# proximity factor: 1 at 0m, ~0 at 2000m+
prox = clip01(1 - (dist / 2000))
mtsa_boost = np.where(df["inside_mtsa"].astype(int) == 1, 1.25, 1.0)

raw_rider = (units ** 0.85) * prox * mtsa_boost
df["transit_rider_impact"] = (100 * minmax(raw_rider)).round(0).astype(int)  # 0–100 index

# Optional: "estimated_new_riders_daily" (still proxy)
# scale based on population increase and proximity (screening-level)
df["estimated_new_riders_daily"] = (df["projected_population_increase"] * prox * 0.25).round(0).astype(int)

# =========================
# 3) Traffic delta (%) (screening proxy)
# =========================
# Units + non-res GFA increase trips; MTSA/transit reduces auto share.
gfa = pd.to_numeric(df["non_residential_gfa_m2"], errors="coerce").fillna(0)
storeys = pd.to_numeric(df["max_storeys"], errors="coerce")

intensity = minmax(units) * 0.6 + minmax(gfa) * 0.3 + minmax(storeys.fillna(storeys.median())) * 0.1

# Base delta in range ~0–25% for demo realism; reduce if MTSA & close transit
base_delta = 25 * intensity
auto_reduction = np.where(df["inside_mtsa"].astype(int) == 1, 0.80, 1.00) * np.where(dist <= 800, 0.85, 1.00)

# add small noise so not perfectly deterministic
noise = np.random.normal(0, 1.5, len(df))

df["traffic_delta_pct"] = np.clip(base_delta * auto_reduction + noise, 0, 30).round(1)

# =========================
# 4) Environmental stress/load change (0–100 index)
# =========================
# Stress increases with intensity; slightly mitigated by being in transit areas (proxy for compact growth)
# We anchor the concept to canopy/urban forest baseline reporting, but we don't have canopy per parcel here.
stress_raw = (minmax(units) * 0.45 +
              minmax(gfa) * 0.35 +
              minmax(storeys.fillna(storeys.median())) * 0.20)

compact_growth_mitigation = np.where(df["inside_mtsa"].astype(int) == 1, 0.90, 1.00)
stress_raw = stress_raw * compact_growth_mitigation

df["environmental_stress_change"] = (100 * stress_raw).round(0).astype(int)

# =========================
# 5) Shadow impact score (0–100 proxy)
# =========================
# Shadow impact scales nonlinearly with height; and is less concerning in MTSA (proxy: more expected tall form)
# This is NOT a real shadow study—it's a screening indicator aligned with the idea that shadow studies exist.
h = storeys.fillna(storeys.median())
shadow_raw = clip01((h - 3) / (40 - 3))  # normalize storeys 3..40
shadow_raw = shadow_raw ** 1.4  # non-linear: taller buildings accelerate impacts
shadow_raw *= np.where(df["inside_mtsa"].astype(int) == 1, 0.85, 1.00)

df["shadow_impact_score"] = (100 * shadow_raw).round(0).astype(int)

# --- Save back ---
df_mock_urban = df

df_mock_urban.head()

,application_id,submission_date,application_status,district_id,proposed_use,max_storeys,residential_units,non_residential_gfa_m2,inside_mtsa,mtsa_corridor,distance_to_transit_m,district_name,approval_score,synthetic_approval,projected_population_increase,transit_rider_impact,estimated_new_riders_daily,traffic_delta_pct,environmental_stress_change,shadow_impact_score
0,SPM 22 107,2022-07-29,Withheld,1.0,General Retail Commercial,2.0,0,158.38,1,Dundas BRT,72,Lakeview / Port Credit,90.0,1,0,0,0,0.3,0,0
1,SP 22 10,2022-01-19,Withheld,5.0,General Retail Commercial,6.0,0,10175.00,0,Not within a Major Transit Station Area.,176,Malton / Northeast,60.0,0,0,0,0,0.0,3,3
2,SP 22 103,2022-07-20,Withheld,1.0,Residential Townhouses,3.0,70,0.00,0,Not within a Major Transit Station Area.,246,Lakeview / Port Credit,65.0,1,185,2,41,1.8,1,0
3,SP 23 2,2023-01-17,Withheld,9.0,Residential Townhouses; Residential Apartments,12.0,212,0.00,0,Not within a Major Transit Station Area.,70,Meadowvale Village,75.0,1,543,6,131,2.7,4,14
4,OZ/OPA 22 5,2022-01-31,Appealed,1.0,Residential Apartments; General Retail Commercial,NaN,42,150.00,1,Hurontario LRT,60,Lakeview / Port Credit,98.0,1,96,2,23,1.3,1,3


In [13]:
df_mock_urban.to_csv("mississauga_planning_mockdata_urbanimpact_v1.csv", index=False)

In [14]:
import numpy as np
import pandas as pd

np.random.seed(42)
df = df_mock.copy()

# safe numeric conversions
df["residential_units"] = pd.to_numeric(df["residential_units"], errors="coerce").fillna(0)
df["non_residential_gfa_m2"] = pd.to_numeric(df["non_residential_gfa_m2"], errors="coerce").fillna(0)
df["max_storeys"] = pd.to_numeric(df["max_storeys"], errors="coerce").fillna(df["max_storeys"].median())
df["distance_to_transit_m"] = pd.to_numeric(df["distance_to_transit_m"], errors="coerce").fillna(df["distance_to_transit_m"].median())
df["inside_mtsa"] = df["inside_mtsa"].astype(int)

# normalized intensity proxy
def normalized(series):
    return (series - series.min()) / (series.max() - series.min() + 1e-9)

intensity = normalized(df["residential_units"]) * 0.5 + normalized(df["non_residential_gfa_m2"]) * 0.3 + normalized(df["max_storeys"]) * 0.2

# 1) property value change (%) – make a realistic range -5% to +25%
df["property_value_change_pct"] = np.clip(
    5 + 30 * intensity - 10 * normalized(df["distance_to_transit_m"]) + np.random.normal(0, 2, len(df)),
    -5, 25
).round(1)

# 2) default risk (0-100)
# higher for high intensity and far from transit
default_risk_raw = intensity * 0.6 + normalized(df["distance_to_transit_m"]) * 0.4
df["default_risk_score"] = (100 * normalized(default_risk_raw)).round(0).astype(int)

# 3) capital concentration risk (0-100)
# higher if gfa and units are large relative to dataset
ccr_raw = normalized(df["non_residential_gfa_m2"]) * 0.5 + normalized(df["residential_units"]) * 0.5
df["capital_concentration_risk"] = (100 * normalized(ccr_raw)).round(0).astype(int)

# 4) construction loan feasibility (0-100)
# good for less intense & inside transit areas
clf_raw = (1 - intensity) * 0.7 + df["inside_mtsa"] * 0.3
df["construction_loan_feasibility"] = (100 * normalized(clf_raw)).round(0).astype(int)

# 5) insurance premium shift index (0-100)
# tie to environmental stress score from earlier and some randomness
env_stress = df.get("environmental_stress_change", normalized(intensity))
ip_raw = normalized(env_stress * 0.8 + normalized(df["distance_to_transit_m"]) * 0.2)
df["insurance_premium_shift_index"] = (100 * ip_raw).round(0).astype(int)

df_financial = df

df_financial[[
    "property_value_change_pct",
    "default_risk_score",
    "capital_concentration_risk",
    "construction_loan_feasibility",
    "insurance_premium_shift_index"
]].head()

,property_value_change_pct,default_risk_score,capital_concentration_risk,construction_loan_feasibility,insurance_premium_shift_index
0,5.7,3,0,100,0
1,4.2,15,5,39,7
2,4.8,18,1,40,5
3,8.8,8,4,37,7
4,4.6,3,1,98,2


In [15]:
df_financial.to_csv(
    "mississauga_planning_mockdata_urban_financial_v1.csv",
    index=False
)

In [16]:
df_full = pd.read_csv("mississauga_planning_mockdata_urban_financial_v1.csv")

df_full.head()

,application_id,submission_date,application_status,district_id,proposed_use,max_storeys,residential_units,non_residential_gfa_m2,inside_mtsa,mtsa_corridor,distance_to_transit_m,district_name,approval_score,synthetic_approval,property_value_change_pct,default_risk_score,capital_concentration_risk,construction_loan_feasibility,insurance_premium_shift_index
0,SPM 22 107,2022-07-29,Withheld,1.0,General Retail Commercial,2.0,0,158.38,1,Dundas BRT,72,Lakeview / Port Credit,90.0,1,5.7,3,0,100,0
1,SP 22 10,2022-01-19,Withheld,5.0,General Retail Commercial,6.0,0,10175.00,0,Not within a Major Transit Station Area.,176,Malton / Northeast,60.0,0,4.2,15,5,39,7
2,SP 22 103,2022-07-20,Withheld,1.0,Residential Townhouses,3.0,70,0.00,0,Not within a Major Transit Station Area.,246,Lakeview / Port Credit,65.0,1,4.8,18,1,40,5
3,SP 23 2,2023-01-17,Withheld,9.0,Residential Townhouses; Residential Apartments,12.0,212,0.00,0,Not within a Major Transit Station Area.,70,Meadowvale Village,75.0,1,8.8,8,4,37,7
4,OZ/OPA 22 5,2022-01-31,Appealed,1.0,Residential Apartments; General Retail Commercial,6.0,42,150.00,1,Hurontario LRT,60,Lakeview / Port Credit,98.0,1,4.6,3,1,98,2


In [26]:
import numpy as np
import pandas as pd

np.random.seed(42)

df = df_financial.copy()

# ---- Ensure needed numeric fields ----
for col in ["residential_units", "non_residential_gfa_m2", "max_storeys", "distance_to_transit_m", "inside_mtsa"]:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

df["residential_units"] = pd.to_numeric(df["residential_units"], errors="coerce").fillna(0)
df["non_residential_gfa_m2"] = pd.to_numeric(df["non_residential_gfa_m2"], errors="coerce").fillna(0)
df["max_storeys"] = pd.to_numeric(df["max_storeys"], errors="coerce").fillna(df["max_storeys"].median())
df["distance_to_transit_m"] = pd.to_numeric(df["distance_to_transit_m"], errors="coerce").fillna(df["distance_to_transit_m"].median())
df["inside_mtsa"] = df["inside_mtsa"].astype(int)

# ---- Helpers ----
def minmax(s):
    s = pd.to_numeric(s, errors="coerce")
    mn, mx = np.nanmin(s), np.nanmax(s)
    return (s - mn) / (mx - mn + 1e-9)

def clip01(x):
    return np.clip(x, 0, 1)

# ---- Core proxies you already have ----
units_n = minmax(df["residential_units"])
gfa_n = minmax(df["non_residential_gfa_m2"])
height_n = minmax(df["max_storeys"])
dist = df["distance_to_transit_m"]

# intensity proxy (bigger projects tend to increase local pressure in a simple demo model)
intensity = 0.55 * units_n + 0.25 * gfa_n + 0.20 * height_n

# transit access proxy: closer transit -> higher access
prox = clip01(1 - (dist / 2000))  # 0..1

# =========================
# 1) Rent increase forecast (%): 0–15%
# =========================
# Anchor: rent pressure exists and varies; we keep within a realistic demo range and add noise. (CMHC context)
# Logic: higher intensity + better transit access often correlates with higher demand/amenity value (proxy),
# but MTSA supply can slightly moderate.
base = 2.5 + 10.0 * (0.6 * intensity + 0.4 * prox)   # roughly 2.5% to 12.5%
supply_mitigation = np.where(df["inside_mtsa"] == 1, -0.8, 0.0)
noise = np.random.normal(0, 1.2, len(df))

df["rent_increase_forecast_pct"] = np.clip(base + supply_mitigation + noise, 0, 15).round(1)

# =========================
# 2) Rent burden score (0–100)
# =========================
# Anchor concept: >30% of income on shelter costs = unaffordable (StatsCan).
# We don't have incomes, so we build a *relative burden risk* using rent pressure + low added supply.
# Logic:
#  - higher rent forecast -> higher burden
#  - fewer units added -> less relief -> higher burden
#  - far from transit -> higher transport costs -> higher combined burden (proxy)
supply_relief = 1 - units_n
transport_cost_proxy = minmax(dist)

burden_raw = (
    0.55 * minmax(df["rent_increase_forecast_pct"]) +
    0.25 * supply_relief +
    0.20 * transport_cost_proxy
)

df["rent_burden_score"] = (100 * burden_raw).round(0).astype(int)

# =========================
# 3) Displacement risk score (0–100)
# =========================
# Anchor: displacement often aligns with affordability pressure + residential instability concepts
# (CIMD has a "residential instability" dimension; we mimic the idea, not the true index).
# Logic:
#  - higher rent burden -> higher displacement pressure
#  - higher rent forecast -> higher pressure
#  - high non-res GFA relative to units can proxy "commercial/upscaling pressure"
gfa_per_unit = df["non_residential_gfa_m2"] / (df["residential_units"] + 1)
gfa_per_unit_n = minmax(gfa_per_unit)

disp_raw = (
    0.50 * minmax(df["rent_burden_score"]) +
    0.30 * minmax(df["rent_increase_forecast_pct"]) +
    0.20 * gfa_per_unit_n
)

df["displacement_risk_score"] = (100 * disp_raw).round(0).astype(int)

# =========================
# 4) Transit speed impact score (0–100)
# =========================

mtsa_boost = np.where(df["inside_mtsa"] == 1, 0.15, 0.0)
speed_raw = clip01(prox + mtsa_boost)

df["transit_speed_impact_score"] = (100 * speed_raw).round(0).astype(int)

# ---- Result preview ----
df_socio = df

df_socio[[
    "rent_increase_forecast_pct",
    "rent_burden_score",
    "displacement_risk_score",
    "transit_speed_impact_score"
]].head()

,rent_increase_forecast_pct,rent_burden_score,displacement_risk_score,transit_speed_impact_score
0,6.2,52,37,100
1,6.1,53,39,91
2,6.8,59,47,88
3,8.4,68,63,96
4,5.4,45,27,100


In [18]:
df_socio.to_csv("mississauga_planning_mockdata_urban_financial_socio_v1.csv", index=False)

## MERGING ALL DATASET INTO 1 DATA

In [19]:
import pandas as pd

base = pd.read_csv("mississauga_planning_mockdata_v1.csv")
urban = pd.read_csv("mississauga_planning_mockdata_urbanimpact_v1.csv")
fin = pd.read_csv("mississauga_planning_mockdata_urban_financial_v1.csv")
socio = pd.read_csv("mississauga_planning_mockdata_urban_financial_socio_v1.csv")

In [20]:
for d in [base, urban, fin, socio]:
    d["application_id"] = d["application_id"].astype(str).str.strip()

In [21]:
urban_cols = ["application_id",
              "projected_population_increase","traffic_delta_pct","environmental_stress_change",
              "shadow_impact_score","transit_rider_impact","estimated_new_riders_daily"]

fin_cols = ["application_id",
            "property_value_change_pct","default_risk_score","capital_concentration_risk",
            "construction_loan_feasibility","insurance_premium_shift_index"]

socio_cols = ["application_id",
              "rent_increase_forecast_pct","rent_burden_score","displacement_risk_score",
              "transit_speed_impact_score"]

In [22]:
df_final = (base
            .merge(urban[urban_cols], on="application_id", how="left")
            .merge(fin[fin_cols], on="application_id", how="left")
            .merge(socio[socio_cols], on="application_id", how="left"))

In [23]:
df_final.shape, df_final["application_id"].nunique()

((209, 29), 209)

In [24]:
df_final.to_csv("mississauga_planning_mockdata_FINAL_v1.csv", index=False)

In [25]:

df_final = pd.read_csv("mississauga_planning_mockdata_FINAL_v1.csv")

df_final.head()

,application_id,submission_date,application_status,district_id,proposed_use,max_storeys,residential_units,non_residential_gfa_m2,inside_mtsa,mtsa_corridor,...,estimated_new_riders_daily,property_value_change_pct,default_risk_score,capital_concentration_risk,construction_loan_feasibility,insurance_premium_shift_index,rent_increase_forecast_pct,rent_burden_score,displacement_risk_score,transit_speed_impact_score
0,SPM 22 107,2022-07-29,Withheld,1.0,General Retail Commercial,2.0,0,158.38,1,Dundas BRT,...,0,5.7,3,0,100,0,6.2,52,37,100
1,SP 22 10,2022-01-19,Withheld,5.0,General Retail Commercial,6.0,0,10175.00,0,Not within a Major Transit Station Area.,...,0,4.2,15,5,39,7,6.1,53,39,91
2,SP 22 103,2022-07-20,Withheld,1.0,Residential Townhouses,3.0,70,0.00,0,Not within a Major Transit Station Area.,...,41,4.8,18,1,40,5,6.8,59,47,88
3,SP 23 2,2023-01-17,Withheld,9.0,Residential Townhouses; Residential Apartments,12.0,212,0.00,0,Not within a Major Transit Station Area.,...,131,8.8,8,4,37,7,8.4,68,63,96
4,OZ/OPA 22 5,2022-01-31,Appealed,1.0,Residential Apartments; General Retail Commercial,NaN,42,150.00,1,Hurontario LRT,...,23,4.6,3,1,98,2,5.4,45,27,100


I intentionally did not build a predictive machine learning model because the objective was not outcome prediction but structured decision support under policy constraints. The dataset used was synthetic and designed to simulate zoning logic, infrastructure thresholds, and planning trade-offs rather than represent historical approval outcomes. Without validated historical approval labels, any supervised model would risk creating artificial predictive confidence. Instead, I focused on building a transparent, rule-based scoring framework (Urban, Financial, and Socioeconomic factors combined into a 0–100 approval score) that aligns with how planners evaluate proposals in practice. This approach prioritizes interpretability, policy alignment, and stakeholder trust over black-box prediction.